## Data loads

In [ ]:
import pandas as pd
import os
from glob import glob

from src.utils.preprocessing import Preprocessor

from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sbn

from collections import Counter
from tqdm import tqdm
import numpy as np


font_fname = os.environ.get('PROJECT_FONT_PATH', '') # point at any local CJK/serif .ttf
font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용

figdir = os.path.join('figure','measure_graph')
os.makedirs(figdir, exist_ok = True)

def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]
measure_dict = dict()

In [ ]:
raw_data_dir = glob(os.path.join('data','reports','*'))
print(raw_data_dir)
df = pd.read_csv(raw_data_dir[0], encoding = 'utf-8-sig', engine = 'c').reset_index()

text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)


for gs, sdf in target_df.groupby(['병원등록번호','수술일자','수술실','진단명','수술명','검사접수일자','검사코드']):
    if sdf.shape[0] > 1:
        print(gs)
        print(sdf.shape)
    # break

### Rule based dataframe

In [ ]:
rule_path_dir = sorted(glob(os.path.join('data','rule_labels', '*labeled.csv')))
print(rule_path_dir)
rule_meta_path, rule_recur_path = rule_path_dir
rule_dict = dict(
    metas = pd.read_csv(rule_meta_path, encoding = 'utf-8-sig', engine = 'c'),
    recur = pd.read_csv(rule_recur_path, encoding = 'utf-8-sig', engine = 'c')
)


### 1st revision dataframe

In [ ]:
# metastasis_positive
revision_01_path_dir = sorted(glob(os.path.join('data','reviewer', '*positive.xlsx')))
print(revision_01_path_dir)
rev_01_meta_path, rev_01_recur_path = revision_01_path_dir
revision_dict = dict()
revision_dict[1] = dict(
    metas = pd.read_excel(rev_01_meta_path, engine = 'openpyxl').dropna(),
    recur = pd.read_excel(rev_01_recur_path, engine = 'openpyxl').dropna()
)


In [ ]:
revision_dict[1]['recur']['label'].value_counts(), revision_dict[1]['metas']['label'].value_counts(),

### 2nd revision dataframe

In [ ]:
revision_02_meta_path_dir = sorted(glob(os.path.join('data','reviewer', '*meta*.xlsx')))
revision_02_recur_path_dir = sorted(glob(os.path.join('data','reviewer', '*recur*.xlsx')))

print(revision_02_meta_path_dir)
print(revision_02_recur_path_dir)
revision_dict[2] = dict(
    metas = pd.concat([pd.read_excel(p, engine = 'openpyxl').dropna() for p in revision_02_meta_path_dir]),
    recur = pd.concat([pd.read_excel(p, engine = 'openpyxl').dropna() for p in revision_02_recur_path_dir])
)

In [ ]:
revision_dict[1]['recur']['label'].value_counts(), revision_dict[1]['metas']['label'].value_counts(),

In [ ]:
revision_dict[2]['recur']['재분류'].value_counts(), revision_dict[2]['metas']['재분류'].value_counts(),

## Duplicate Check

### Def function

In [ ]:
def agg_df_load(target):
    data_dict = preprocessor.target_filtering(target)
    merged_target_df = pd.concat([check_label(d, k) for k, d in data_dict.items()]).sort_index()
    neg_length = df.shape[0] - merged_target_df.shape[0]

    temp_df = df.copy()
    temp_df.loc[merged_target_df.index, f'{target}_text'] = merged_target_df['검사결과결론내용']
    temp_df.loc[merged_target_df.index, f'label'] = merged_target_df['label']
    target_df = temp_df.iloc[merged_target_df.index].copy()
    drop_df = temp_df.loc[~temp_df.index.isin(target_df.index)]

    df1 = target_df.reset_index()[['index','검사결과결론내용',target+'_text', 'label']].set_axis(['index','raw_text','prep_text','raw_label'], axis = 1)
    df2 = rule_dict[target][['index','검사결과결론내용','label']].set_axis(['index','rule_text','rule_label'], axis = 1)
    df3 = revision_dict[1][target].set_axis(['index','R1_text','R1_label'], axis = 1)
    df4 = revision_dict[2][target][['index','검사결과결론내용','재분류']].set_axis(['index','R2_text','R2_label'], axis = 1)

    print(df1.shape, df2.shape, df3.shape, df4.shape)

    agg_df = df1.merge(df2, on = 'index', how = 'outer')
    agg_df = agg_df.merge(df3, on = 'index', how = 'outer')
    agg_df = agg_df.merge(df4, on = 'index', how = 'outer')

    agg_df['n_samples'] = agg_df.groupby('prep_text')['prep_text'].transform('count')
    return dict(
        agg = agg_df, 
        drop = drop_df,
        total_length = len(df), 
        neg_length = neg_length, 
        R1 = df3,
        R2 = df4,
    )

    
def vis_confusion(cm, labels, save_strs, figsize = (7, 6), fm_size = 15):
    
    plt.rcParams['font.size'] = fm_size
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    sbn.heatmap(
        cm, 
        annot=True, fmt='d', cmap='Blues',
        xticklabels=[i.capitalize() for i in labels],
        yticklabels=[i.capitalize() for i in labels], 
        ax=ax, 
        cbar = False
    )
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    fig.tight_layout()

    # target, rev_ver, pred_ver = save_strs 
    # fig.savefig(os.path.join(figdir, f'{target}_{rev_ver}_{pred_ver}.png'), dpi = 400)
    fig.savefig(os.path.join(figdir, '_'.join(save_strs) + '.png'), dpi = 400)
    plt.close()


def score2dict(yt, yp, n_samples = None):
    return dict(
        accuracy = accuracy_score(yt, yp, sample_weight = n_samples),
        f1 = f1_score(yt, yp, average = 'macro', sample_weight = n_samples),
        weighted_f1 = f1_score(yt, yp, average = 'weighted', sample_weight = n_samples),
    )


def sum_sub_return(sub_df, dup_df, target_col, flag_dup = False):
    label_col = target_col + '_label'
    text_col = target_col + '_text'
    # if flag_dup:
    #     sub_df
    if flag_dup:
        dup_df = dup_df.drop_duplicates(text_col)
        sub_df = sub_df.drop_duplicates(text_col)
        sub_df = sub_df.loc[~sub_df.prep_text.isin(dup_df.prep_text)]


    sum_sub_df = pd.concat([
        sub_df[label_col].value_counts().loc[labels],
        dup_df[label_col].value_counts().loc[labels],
    ], axis = 1)
    sum_sub_df.loc[:, 'Sum'] = sum_sub_df.sum(1)
    sum_sub_df.loc['Sum'] = sum_sub_df.sum(0)
    
    return (sub_df, dup_df), sum_sub_df.set_axis(['Uni.','D.C.','Sum'], axis = 1)

In [ ]:
subs = []
raw_df_dict = dict()
for target in ['recur','metas']:
    labels = ['positive','uncertain','negative'] if 'recur' == target else ['positive','uncertain','regional','negative']
    agg_dict = agg_df_load(target)
    agg_df = agg_dict.pop('agg')
    drop_df = agg_dict.pop('drop')
    r1_df = agg_dict.pop('R1')
    r2_df = agg_dict.pop('R2')

    dup_index = agg_df.R1_label.notna() & agg_df.R2_label.notna()
    cover_index = agg_df.R1_label.notna() | agg_df.R2_label.notna()
    error_idx = agg_df.loc[dup_index].R1_label != agg_df.loc[dup_index].R2_label

    dup_df = agg_df.loc[dup_index]
    r1_only_df = agg_df.loc[agg_df.R1_label.notna() & agg_df.R2_label.isna()]
    r2_only_df = agg_df.loc[agg_df.R1_label.isna() & agg_df.R2_label.notna()]

    temp = []
    raw_df_dict[target] = dict()
    for dup_flag in [True, False]:
        r1_dfs, r1_score = sum_sub_return(r1_only_df, dup_df, 'R1', dup_flag)
        r2_dfs, r2_score = sum_sub_return(r2_only_df, dup_df, 'R2', dup_flag)
        temp.append(pd.concat([r1_score, r2_score], axis = 1))
        raw_df_dict[target][dup_flag] = dict(r1 = r1_dfs, r2 = r2_dfs)
    sub_count_df = pd.concat(temp, axis = 1)
    subs.append(sub_count_df)


In [ ]:
from datetime import datetime
today = datetime.now().date().isoformat()
log_save_dir = os.path.join('outputs','logs',today)
os.makedirs(log_save_dir, exist_ok=True)


In [ ]:
new_cols = pd.MultiIndex.from_product([['Drop dup.','Raw'], ['Rev. 1', 'Rev 2'], ['Uni.','D.C.','Sum']])
concat_sub = pd.concat(subs).set_axis(new_cols, axis=1)
concat_sub.index = [i.capitalize() for i in concat_sub.index]

In [ ]:
concat_sub.to_excel(os.path.join(log_save_dir, 'Count_samples.xlsx'), index = True)